# News Collection via GNews API

Queries the [GNews](https://gnews.io) API for several SAP / enterprise-software search terms, deduplicates the results (exact + fuzzy title matching), and saves them to `data/gnews_articles.json`.

In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd
import requests
from rapidfuzz import fuzz

sys.path.append("..")
from API_Key import gnews_API

## Define search queries

In [ ]:
API_KEY = gnews_API

queries = [
    "SAP",
    "SAP AI",
    "SAP ERP",
    "enterprise software",
    "business AI",
    "SAP Technologies",
]

## Fetch articles for each query

In [ ]:
all_articles = []

for q in queries:
    url = (
        "https://gnews.io/api/v4/search?"
        f"q={q}&lang=en&max=10&apikey={API_KEY}"
    )

    response = requests.get(url)
    time.sleep(0.2)  # be polite to the API rate limit
    data = response.json()

    if "articles" in data:
        all_articles.extend(data["articles"])
    else:
        print(f"Query failed: {q}")
        print(data)

print(f"Collected {len(all_articles)} raw articles")

## Build a DataFrame

Uses `.get()` everywhere so a missing field in the API response doesn't crash the whole pipeline.

In [ ]:
gnews_docs = [
    {
        "title": article.get("title", ""),
        "description": article.get("description", ""),
        "content": article.get("content", ""),
        "source": article.get("source", {}).get("name", ""),
        "published": article.get("publishedAt", ""),
        "url": article.get("url", ""),
    }
    for article in all_articles
]

gnews_df = pd.DataFrame(gnews_docs)
print(f"Built DataFrame with {len(gnews_df)} rows")

## Remove exact duplicate titles

In [ ]:
print("Before:", len(gnews_df))

gnews_df = gnews_df.drop_duplicates(subset=["title"]).reset_index(drop=True)

print("After exact dedup:", len(gnews_df))
gnews_df.head(1)

## Remove near-duplicate titles (fuzzy matching)

Some outlets republish the same story under slightly different headlines. Titles with a similarity score ≥ 80 are treated as duplicates; the first occurrence is kept.

In [ ]:
titles = gnews_df["title"].fillna("").tolist()
to_drop = set()

for i in range(len(titles)):
    if i in to_drop:
        continue
    for j in range(i + 1, len(titles)):
        if j in to_drop:
            continue
        if fuzz.ratio(titles[i], titles[j]) >= 80:
            to_drop.add(j)

gnews_df = gnews_df.drop(index=list(to_drop)).reset_index(drop=True)

print("After fuzzy dedup:", len(gnews_df))

## Save to JSON

In [ ]:
DATA_DIR = Path("..") / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

output_path = DATA_DIR / "gnews_articles.json"
gnews_df.to_json(output_path, orient="records", indent=2)

print(f"Saved {len(gnews_df)} articles to {output_path}")